# Task 2 — Transform Data

**Purpose:** second task in the Workflow. Runs only after `01_ingest_data` succeeds (this dependency is configured in the Job UI, not in code).

**Concept demoed:** multi-task dependency + reading a value passed from the previous task.

In [ ]:
dbutils.widgets.text("catalog", "main", "Target catalog")
dbutils.widgets.text("schema", "default", "Target schema")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

# Pull the table name produced by task 1. If run standalone (no upstream
# task ran), fall back to the default naming convention.
try:
    ingested_table = dbutils.jobs.taskValues.get(
        taskKey="ingest_data", key="ingested_table",
        default=f"{catalog}.{schema}.demo_trips_ingested"
    )
except Exception:
    ingested_table = f"{catalog}.{schema}.demo_trips_ingested"

print(f"Reading from: {ingested_table}")

In [ ]:
from pyspark.sql import functions as F

df = spark.table(ingested_table)

summary = (
    df.groupBy("pickup_zip")
      .agg(
          F.count("*").alias("trip_count"),
          F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
          F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
      )
      .orderBy(F.desc("trip_count"))
)

display(summary.limit(10))

In [ ]:
target_table = f"{catalog}.{schema}.demo_trips_summary"
summary.write.mode("overwrite").saveAsTable(target_table)
print(f"Wrote summary to {target_table}")

dbutils.jobs.taskValues.set(key="summary_table", value=target_table)